In [ ]:
import time
import sys
import os

import joblib
import pandas as pd
from tslearn.metrics import cdist_dtw

sys.path.insert(0, '../../..') # Load Parent directory to import utils
from utils.constants import Configs
from utils.helper_functions import normalize_series # calcula a distância entre séries temporais usando DTW

### Load Data From cache

In [2]:
cache_path = Configs.ICATER_ALL_SENSORS_FILTERED_DATA_DIR

if os.path.exists(cache_path):
    df = pd.read_csv(cache_path)
else:
    print(f"Processed data not found at {cache_path}. Please run the data processing notebook first.")
    raise FileNotFoundError(f"Processed data not found at {cache_path}. Please run the data processing notebook first.")

### Calculate DTW Matrix for all movements and save on cache

In [4]:
group_array = ['xs', 'ys', 'zs', 'xe', 'ye', 'ze', 'xw', 'yw', 'zw']
sample_size = None

times = []
for movement in ["Pick", "Move", "Place"]:
    print(f"Calculating DTW distance matrix for {movement} movements...")

    cache_name = Configs.all_sensors_dtw_distance_matrix(movement)

    normalized_data = normalize_series(
            df_input=df,
            movLabel_filter=movement,
            useScaler=False,
            group_array=group_array,
            sample_size=sample_size
        )  # shape (n_samples, T, n_features)

    # Calculate DTW distance matrix
    start_time = time.perf_counter()
    dtw_matrix = cdist_dtw(normalized_data, n_jobs=-1) 
    end_time = time.perf_counter()

    # Save the time taken for this movement
    elapsed_time = end_time - start_time
    times.append(elapsed_time)
    print(f"DTW distance matrix for {movement} calculated in {elapsed_time:.2f} seconds.")

    # Save to cache
    joblib.dump(dtw_matrix, cache_name)

Calculating DTW distance matrix for Pick movements...


Processing time series:   0%|          | 0/380 [00:00<?, ?it/s]

Processing time series: 100%|██████████| 380/380 [00:00<00:00, 3756.79it/s]

Shape of first series: (111, 9)


Shape of formatted series: (380, 241, 9)
DTW distance matrix for Pick calculated in 49.17 seconds.
Calculating DTW distance matrix for Move movements...


Processing time series: 100%|██████████| 95/95 [00:00<00:00, 4184.53it/s]

Shape of first series: (72, 9)
Shape of formatted series: (95, 196, 9)


DTW distance matrix for Move calculated in 2.35 seconds.
Calculating DTW distance matrix for Place movements...


Processing time series: 100%|██████████| 380/380 [00:00<00:00, 2025.29it/s]

Shape of first series: (290, 9)


Shape of formatted series: (380, 290, 9)
DTW distance matrix for Place calculated in 57.64 seconds.


In [5]:
times

[49.17131296100001, 2.350784327999918, 57.63818901700006]

In [7]:
times_dict = {
    "Pick": times[0],
    "Move": times[1],
    "Place": times[2]
}
joblib.dump(times_dict, Configs.ALL_SENSORS_DTW_CALCULATION_TIMES_CACHE)

['/home/hugo-guimar-es/Documents/GitHub/python_zeromq/Outlier_Removal/ICater/ICater_Shoulder_Elbow_Hand/cache/metadata/all_sensors_dtw/all_sensors_dtw_calculation_times.pkl']